Импортируем Библиотеки

In [1]:
import pandas as pd
import numpy as np
from numba import njit
from tabulate import tabulate

Функция для нахождения меньшей строки (по условию доминантности всех принадлежащих элементов)

In [ ]:
@njit
def is_strictly_less(data_i, data_j):
    return np.all(data_i < data_j)

Основная функция для нахождения доминирующих и доминируемых строк в датасете (в нашем случае, устанавливаем индексацию на колонку "molecule_id")

In [ ]:
def find_strictly_less_relationships(file_path):
    try:
        df = pd.read_csv(file_path, index_col='molecule_id')
    except KeyError:
        raise ValueError("Column is missing in the dataset.")
    
    molecule_ids = df.index.values.flatten()
    
    numeric_columns = [col for col in df.columns if df[col].dtype in [np.int64, np.int32]]
    df = df[numeric_columns]
    
    df.fillna(0, inplace=True)
    
    data = df.to_numpy()
    n, m = data.shape
    
    strictly_less_map = {molecule_id: [] for molecule_id in molecule_ids}
    
    for i in range(n):
        current_id = molecule_ids[i]
        
        for j in range(n):
            if i == j:
                continue
            
            if is_strictly_less(data[i], data[j]):
                strictly_less_map[current_id].append(molecule_ids[j])
    
    result = {molecule_id: dominators for molecule_id, dominators in strictly_less_map.items() if dominators}
    
    return result

Вывод результатов

In [4]:
if __name__ == "__main__":
    file_path = "./generated_dataset.csv"
    try:
        strictly_less_relationships = find_strictly_less_relationships(file_path)
        print("Strictly less relationships:")
        for dominated_id, dominators in strictly_less_relationships.items():
            print(f"Row {dominated_id} is strictly less than rows: {dominators}")
    except Exception as e:
        print(f"Error: {e}")

Strictly less relationships:
Row 53 is strictly less than rows: [6350, 9314]
Row 89 is strictly less than rows: [5126]
Row 163 is strictly less than rows: [2932]
Row 185 is strictly less than rows: [7862]
Row 239 is strictly less than rows: [6326]
Row 298 is strictly less than rows: [6326]
Row 465 is strictly less than rows: [675]
Row 622 is strictly less than rows: [2068]
Row 686 is strictly less than rows: [3928]
Row 992 is strictly less than rows: [6326]
Row 1025 is strictly less than rows: [4449]
Row 1461 is strictly less than rows: [5308]
Row 1541 is strictly less than rows: [9554]
Row 1860 is strictly less than rows: [3444]
Row 2079 is strictly less than rows: [6326]
Row 2114 is strictly less than rows: [5583, 6902]
Row 2128 is strictly less than rows: [6326, 7125]
Row 2572 is strictly less than rows: [1383, 4307, 9606]
Row 2632 is strictly less than rows: [5636]
Row 2654 is strictly less than rows: [3449, 6788]
Row 3094 is strictly less than rows: [675, 676]
Row 3165 is strictly

Берём выборочно строки из датасета и визуально определяем правильность выполнения алгоритма

In [5]:
df = pd.read_csv("./generated_dataset.csv")
target_ids = [53, 6350]
filtered_df = df[df['molecule_id'].isin(target_ids)]

print("\nНовый датасет (красивый вид):")
print(tabulate(filtered_df, headers='keys', tablefmt='psql', showindex=False))


Новый датасет (красивый вид):
+---------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+
|   molecule_id |   Column_1 |   Column_2 |   Column_3 |   Column_4 |   Column_5 |   Column_6 |   Column_7 |   Column_8 |   Column_9 |   Column_10 |   Column_11 |   Column_12 |   Column_13 |   Column_14 |   Column_15 |   Column_16 |   Column_17 |   Column_18 |   Column_19 |   Column_20 |
|---------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------|
|            53 |         14 |         64 |         38 |         37 |         11 |         77 |       